In [8]:
import gluonts
from gluonts.dataset.repository import get_dataset, dataset_recipes
from gluonts.dataset.multivariate_grouper import MultivariateGrouper
from gluonts.dataset.repository.datasets import dataset_recipes, get_dataset
from gluonts.evaluation.backtest import make_evaluation_predictions
from gluonts.evaluation import MultivariateEvaluator
from matplotlib import pyplot as plt
import matplotlib.dates as mdates
from typing import List, Optional, Callable, Iterable
from itertools import islice
import pandas as pd
import numpy as np
import jax
import jax.random as random
import jax.numpy as jnp
import flax.nnx as nnx
import sys
sys.path.append('../')
sys.path.append('..')
from data.dataloader import GluonTSDataLoaderAdapter
from data.gluonts_loaders import get_dataset_loader
from models.flow import Flow
from trainers.flow_eval import FlowEvaluator
from gluonts.model.predictor import Predictor, OutputTransform
from gluonts.model.forecast_generator import (
    ForecastGenerator,
    SampleForecastGenerator,
    to_numpy,
)
from gluonts.transform import SelectFields, Transformation

from typing import Iterator, List, Optional, Union, Any
from gluonts.model.forecast import Forecast, SampleForecast
from gluonts.dataset.common import Dataset
from gluonts.dataset.loader import TrainDataLoader, InferenceDataLoader, ValidationDataLoader
from gluonts.transform import AddObservedValuesIndicator, InstanceSplitter, ExpectedNumInstanceSampler, TestSplitSampler, ValidationSplitSampler
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddObservedValuesIndicator,
    InstanceSplitter,
    ExpectedNumInstanceSampler,
    TestSplitSampler,
    Transformation,
    Chain,
    InstanceSampler,
    SelectFields,
)

from data.gluonts_loaders import batchify, _to_dataframe

In [9]:
tain_loader, val_loader, test_data, dims, scaler = get_dataset_loader('electricity_nips', 32, 2)
rngs = nnx.Rngs(0)
rngs_model = nnx.Rngs(rngs())
rngs_train = nnx.Rngs(rngs()) 
model = Flow(
    dim_x=dims["data_dim"],
    dim_c=dims["cov_dim"],
    dim_h=32,
    dim_t=4,
    n_rnn_layers=1,
    rnn_hidden=64,
    residual_layers=2,
    residual_channels=8,
    residual_hidden=32,
    rngs=rngs_model,
    scaler=scaler
)
    

In [42]:
test_data.prediction_length

24

In [28]:
import flax.nnx as nnx
import jax
import jax.random as jrandom

class JaxPredictor(Predictor):
    def __init__(
        self,
        config,
        input_names: List[str],
        required_fields: List[str], 
        prediction_net: nnx.Module,
        batch_size: int,
        prediction_length: int,
        context_length: int,
        scaler: Any = None,
        rngs: Any = None,
    ) -> None:
        self.config = config
        self.input_names = input_names
        self.batch_size = batch_size
        self.scaler = scaler
        self.prediction_length = prediction_length
        self.context_length = context_length
        self.prediction_net = prediction_net
        self.required_fields = required_fields
        self.rngs = rngs    

    @property
    def network(self):
        return self.prediction_net
    
    def predict(self, dataset: Dataset) -> Iterator[Forecast]:

        test_splitter = InstanceSplitter(
            target_field=FieldName.TARGET,
            is_pad_field=FieldName.IS_PAD,
            start_field=FieldName.START,
            forecast_start_field=FieldName.FORECAST_START,
            instance_sampler=TestSplitSampler(),
            past_length=self.context_length,
            future_length=self.prediction_length,
            time_series_fields=[FieldName.FEAT_TIME],
        ) 

        inference_data_loader = InferenceDataLoader(dataset=dataset.input,
                                        transform=test_splitter + SelectFields(self.input_names + self.required_fields, allow_missing=True),
                                        batch_size=self.batch_size,
                                        stack_fn=batchify)
    
        keys = jrandom.split(self.rngs(), self.config["inference"]["mcmc_samples"])
        evaluator = FlowEvaluator(self.prediction_net, nfe=self.config["inference"]["nfe"])

        for batch in inference_data_loader:
            print(batch["past_target"].shape)
            history = jnp.array(batch["past_target"])
            future_covariates = jnp.array(batch["future_time_feat"])
            past_covariates = jnp.array(batch["past_time_feat"])
            covariates = jnp.concatenate([past_covariates, future_covariates], axis=1)
            start_date = batch["forecast_start"]
            predictions = jax.vmap(jax.vmap(evaluator.predict_trajectory, in_axes=(0,None,None,None)),in_axes=(None,0,0,None))(keys, history, covariates, self.prediction_length)
            predictions = np.array(predictions)
            for i in range(predictions.shape[0]):
                yield SampleForecast(
                    samples = predictions[i],
                    start_date=start_date[i],
                    item_id=None,
                    info={}
                )
            





In [29]:
import yaml

In [30]:
config = yaml.safe_load(open('../configs/base_config.yaml', 'r'))
input_names = ["past_target"]
required_fields = ["forecast_start", "future_time_feat", "past_time_feat", "item_id"]
prediction_net = model
batch_size = 32
prediction_length = 24
context_length = 48

predictor = JaxPredictor(config, input_names, required_fields, prediction_net, batch_size, prediction_length, context_length, scaler, rngs_model)


In [33]:
forecast_it = predictor.predict(test_data)
gt_it = map(_to_dataframe, test_data)


In [32]:
print(next(forecast_it))

(7, 48, 370)
gluonts.model.forecast.SampleForecast(info={}, item_id=None, samples=array([[[-0.4560659 ,  0.2525146 ,  0.7948707 , ..., -0.07329428,
          0.07895266, -1.0471355 ],
        [-0.52593637,  0.23041706,  0.79518294, ..., -0.04656154,
          0.08213161, -1.106722  ],
        [-0.5149869 ,  0.24583387,  0.8215536 , ..., -0.05443294,
          0.08435317, -1.1121576 ],
        ...,
        [-0.507637  ,  0.23312272,  0.7849947 , ..., -0.0503842 ,
          0.06755921, -1.0787219 ],
        [-0.5170617 ,  0.22790483,  0.7843776 , ..., -0.06481649,
          0.05744256, -1.0888652 ],
        [-0.4754548 ,  0.2474194 ,  0.81451494, ...,  0.06555261,
          0.23604296, -1.0550998 ]],

       [[-0.47580943, -1.2205533 ,  1.5521741 , ...,  0.2685932 ,
          1.0147481 ,  0.7874375 ],
        [-0.52568626, -1.2448721 ,  1.559192  , ...,  0.27437398,
          0.99961305,  0.7497641 ],
        [-0.49122483, -1.2359885 ,  1.5638176 , ...,  0.2735654 ,
          1.00297   ,

In [34]:
forecasts = list(forecast_it)


(7, 48, 370)


NameError: name 'ts_it' is not defined

In [35]:
targets = list(gt_it)

In [36]:
print(targets[0], forecasts[0])

                         0           1          2           3           4    \
2014-03-19 09:00  141.705063  153.191483  67.242355  133.418579  104.571068   
2014-03-19 10:00  141.129028  183.510635  67.808609  145.566925  105.979965   
2014-03-19 11:00  119.815666  153.723404  77.576447  150.895142  104.571068   
2014-03-19 12:00  139.400925  181.914886  76.019249  155.370850  102.692551   
2014-03-19 13:00  125.576035  136.170212  70.781425  156.862747   97.056984   
...                      ...         ...        ...         ...         ...   
2014-09-01 20:00  107.718895  152.659576  91.449600   99.744247   91.891045   
2014-09-01 21:00  112.903229  164.893616  84.796150   95.055412   95.178459   
2014-09-01 22:00   71.428574  155.319153  69.648926   88.874680   94.239197   
2014-09-01 23:00   58.755760  138.297867  50.537937   66.922424   95.178459   
2014-09-02 00:00   50.691246  115.425529  45.724800   60.315430   86.255478   

                        5           6           7  

In [39]:
evaluator_ = MultivariateEvaluator(quantiles=(np.arange(20)/20.0)[1:], 
                                  target_agg_funcs={'sum': np.sum})
agg_metric, item_metrics = evaluator_(targets, forecasts, num_series=len(test_data))


Running evaluation: 0it [00:00, ?it/s]

Running evaluation: 7it [00:00, 70.40it/s]
Running evaluation: 7it [00:00, 172.85it/s]
Running evaluation: 7it [00:00, 162.47it/s]
Running evaluation: 7it [00:00, 171.29it/s]
Running evaluation: 7it [00:00, 177.52it/s]
Running evaluation: 7it [00:00, 170.28it/s]
Running evaluation: 7it [00:00, 180.48it/s]
Running evaluation: 7it [00:00, 165.63it/s]
Running evaluation: 7it [00:00, 179.05it/s]
Running evaluation: 7it [00:00, 176.23it/s]
Running evaluation: 7it [00:00, 176.30it/s]
Running evaluation: 7it [00:00, 175.53it/s]
Running evaluation: 7it [00:00, 177.40it/s]
Running evaluation: 7it [00:00, 176.51it/s]
Running evaluation: 7it [00:00, 181.73it/s]
Running evaluation: 7it [00:00, 181.54it/s]
Running evaluation: 7it [00:00, 172.87it/s]
Running evaluation: 7it [00:00, 181.06it/s]
Running evaluation: 7it [00:00, 176.15it/s]
Running evaluation: 7it [00:00, 175.65it/s]
Running evaluation: 7it [00:00, 151.99it/s]
Running evaluation: 7it [00:00, 94.85it/s]
Running evaluation: 7it [00:00, 16

In [40]:
print("CRPS:", agg_metric["mean_wQuantileLoss"])
print("ND:", agg_metric["ND"])
print("NRMSE:", agg_metric["NRMSE"])
print("")
print("CRPS-Sum:", agg_metric["m_sum_mean_wQuantileLoss"])
print("ND-Sum:", agg_metric["m_sum_ND"])
print("NRMSE-Sum:", agg_metric["m_sum_NRMSE"])

CRPS: 0.9993707771747188
ND: 0.9999813255443063
NRMSE: 6.872414983093651

CRPS-Sum: 0.9999240356146385
ND-Sum: 0.9999704100403155
NRMSE-Sum: 1.0639794831129668
